# Using the DynamicWrapper Module in baseobjects

## Introduction

The `DynamicWrapper` class provides a flexible way to wrap objects, allowing you to access their attributes and methods through the wrapper. It works by overriding the `__getattr__`, `__setattr__`, and `__delattr__` methods to check wrapped objects when an attribute is not found in the wrapper itself.

This tutorial will guide you through:
- Understanding the purpose and functionality of the `DynamicWrapper` class
- Learning how to create and use a basic `DynamicWrapper`
- Creating custom `DynamicWrapper` subclasses
- Understanding attribute resolution in `DynamicWrapper`
- Exploring advanced features and use cases
- Performance considerations when using `DynamicWrapper`

**Prerequisites:**
- Basic understanding of Python classes and object-oriented programming
- Familiarity with Python's attribute access mechanisms

### Table of Contents

- [Importing the Module](#Importing-the-Module)
- [Core Functionality](#Core-Functionality)
- [Module Interaction](#Module-Interaction)
- [Advanced Features](#Advanced-Features)
- [Examples](#Examples)
- [API Highlights](#API-Highlights)
- [Troubleshooting / FAQs](#Troubleshooting-/-FAQs)
- [Conclusion and Next Steps](#Conclusion-and-Next-Steps)

## Importing the Module

In [14]:
from baseobjects.wrappers import DynamicWrapper

## Core Functionality

The `DynamicWrapper` class is designed to provide a flexible way to wrap objects and access their attributes and methods through the wrapper. It's part of the `baseobjects.wrappers` package and inherits from `BaseObject`.

### Basic Concept

`DynamicWrapper` works by overriding the attribute access methods (`__getattr__`, `__setattr__`, and `__delattr__`) to check wrapped objects when an attribute is not found in the wrapper itself. This allows you to access attributes and methods of wrapped objects as if they were attributes and methods of the wrapper.

Let's create some simple objects to wrap:

In [15]:
class SimpleObject:
    """A simple object to be wrapped."""
    
    def __init__(self, value=0):
        self.value = value
        self.name = "SimpleObject"
    
    def get_value(self):
        """Get the value."""
        return self.value
    
    def set_value(self, value):
        """Set the value."""
        self.value = value
    
    def __str__(self):
        return f"{self.name}(value={self.value})"


class ComplexObject:
    """A more complex object to be wrapped."""
    
    def __init__(self, items=None):
        self.items = items or []
        self.name = "ComplexObject"
    
    def add_item(self, item):
        """Add an item to the list."""
        self.items.append(item)
    
    def get_items(self):
        """Get all items."""
        return self.items
    
    def clear_items(self):
        """Clear all items."""
        self.items = []
    
    def __str__(self):
        return f"{self.name}(items={self.items})"

### Creating a Basic DynamicWrapper

To use `DynamicWrapper`, you need to create a subclass and define the `_wrapped_map_` class attribute, which is a list of attribute names that will contain the objects to wrap.

In [16]:
class MyWrapper(DynamicWrapper):
    """A simple DynamicWrapper subclass."""
    
    # Define which attributes contain objects to wrap
    _wrapped_map_ = ["obj1", "obj2"]

# Create instances of the objects to wrap
simple = SimpleObject(10)
complex = ComplexObject([1, 2, 3])

# Create an instance of the wrapper
wrapper = MyWrapper()
wrapper.obj1 = simple
wrapper.obj2 = complex

print(f"Created wrapper with two objects:")
print(f"  simple: {simple}")
print(f"  complex: {complex}")

Created wrapper with two objects:
  simple: SimpleObject(value=10)
  complex: ComplexObject(items=[1, 2, 3])


### Accessing Wrapped Object Attributes

Once you have created a wrapper and assigned objects to the attributes listed in `_wrapped_map_`, you can access the attributes and methods of those objects through the wrapper.

In [17]:
# Access wrapped object attributes
print("Accessing wrapped object attributes:")
print(f"  wrapper.value: {wrapper.value}")  # From SimpleObject
print(f"  wrapper.name: {wrapper.name}")    # From SimpleObject (first in _wrapped_map_)
print(f"  wrapper.items: {wrapper.items}")  # From ComplexObject

Accessing wrapped object attributes:
  wrapper.value: 10
  wrapper.name: SimpleObject
  wrapper.items: [1, 2, 3]


### Calling Wrapped Object Methods

You can also call methods of wrapped objects through the wrapper.

In [18]:
# Call wrapped object methods
print("Calling wrapped object methods:")
print(f"  wrapper.get_value(): {wrapper.get_value()}")  # From SimpleObject

# Modify the value through the wrapper
wrapper.set_value(20)
print(f"  After wrapper.set_value(20), wrapper.value: {wrapper.value}")
print(f"  simple.value: {simple.value}")  # The original object is modified

# Add an item through the wrapper
wrapper.add_item(4)
print(f"  After wrapper.add_item(4), wrapper.items: {wrapper.items}")
print(f"  complex.items: {complex.items}")  # The original object is modified

Calling wrapped object methods:
  wrapper.get_value(): 10
  After wrapper.set_value(20), wrapper.value: 20
  simple.value: 20
  After wrapper.add_item(4), wrapper.items: [1, 2, 3, 4]
  complex.items: [1, 2, 3, 4]


### Attribute Resolution Order

When you access an attribute through the wrapper, it first checks if the attribute exists in the wrapper itself. If not, it checks each wrapped object in the order they appear in `_wrapped_map_`.

In [19]:
# Create objects with overlapping attribute names
obj1 = SimpleObject(10)
obj1.shared_attr = "from obj1"

obj2 = ComplexObject([1, 2, 3])
obj2.shared_attr = "from obj2"
obj2.unique_attr = "only in obj2"

# Create a wrapper
class AttributeWrapper(DynamicWrapper):
    _wrapped_map_ = ["first", "second"]

wrapper = AttributeWrapper()
wrapper.first = obj1
wrapper.second = obj2

print("Created wrapper with objects having overlapping attributes:")
print(f"  obj1.shared_attr: {obj1.shared_attr}")
print(f"  obj2.shared_attr: {obj2.shared_attr}")

# Access attributes - first wrapped object takes precedence
print("\nAccessing attributes (first wrapped object takes precedence):")
print(f"  wrapper.shared_attr: {wrapper.shared_attr}")  # From obj1
print(f"  wrapper.unique_attr: {wrapper.unique_attr}")  # From obj2

# Change the order of wrapped objects
print("\nChanging the order of wrapped objects:")
wrapper = AttributeWrapper()
wrapper.first = obj2
wrapper.second = obj1

print(f"  wrapper.shared_attr: {wrapper.shared_attr}")  # Now from obj2

Created wrapper with objects having overlapping attributes:
  obj1.shared_attr: from obj1
  obj2.shared_attr: from obj2

Accessing attributes (first wrapped object takes precedence):
  wrapper.shared_attr: from obj1
  wrapper.unique_attr: only in obj2

Changing the order of wrapped objects:
  wrapper.shared_attr: from obj2


## Module Interaction

The `DynamicWrapper` class is part of the `baseobjects.wrappers` package, which also includes other wrapper classes like `StaticWrapper`. Let's explore how `DynamicWrapper` interacts with other components of the package.

### Relationship with BaseObject

`DynamicWrapper` inherits from `BaseObject`, which is the base class for most objects in the baseobjects package. This provides a consistent interface and behavior across the package.

In [20]:
from baseobjects.bases import BaseObject

# Check inheritance
print(f"DynamicWrapper is a subclass of BaseObject: {issubclass(DynamicWrapper, BaseObject)}")

DynamicWrapper is a subclass of BaseObject: True


## Advanced Features

Now let's explore some advanced features and use cases of `DynamicWrapper`.

### Creating a Custom DynamicWrapper Subclass

You can create a custom `DynamicWrapper` subclass with additional functionality:

In [21]:
class CustomDynamicWrapper(DynamicWrapper):
    """A custom DynamicWrapper that wraps both simple and complex objects."""
    
    # Define which attributes contain objects to wrap
    _wrapped_map_ = ["simple", "complex"]
    
    def __init__(self, simple_obj=None, complex_obj=None):
        """Initialize the wrapper with simple and complex objects."""
        super().__init__()
        self.simple = simple_obj or SimpleObject()
        self.complex = complex_obj or ComplexObject()
    
    def get_combined_str(self):
        """Get a string representation of both wrapped objects."""
        return f"Combined: {self.simple}, {self.complex}"

# Create a custom wrapper
wrapper = CustomDynamicWrapper(
    SimpleObject(5),
    ComplexObject([10, 20, 30])
)

print("Created custom wrapper:")
print(f"  wrapper.get_combined_str(): {wrapper.get_combined_str()}")

# Access and modify wrapped object attributes
print("\nAccessing and modifying wrapped object attributes:")
print(f"  Initial wrapper.value: {wrapper.value}")
wrapper.value = 15
print(f"  After wrapper.value = 15: {wrapper.value}")

print(f"  Initial wrapper.items: {wrapper.items}")
wrapper.add_item(40)
print(f"  After wrapper.add_item(40): {wrapper.items}")

Created custom wrapper:
  wrapper.get_combined_str(): Combined: SimpleObject(value=5), ComplexObject(items=[10, 20, 30])

Accessing and modifying wrapped object attributes:
  Initial wrapper.value: 5
  After wrapper.value = 15: 15
  Initial wrapper.items: [10, 20, 30]
  After wrapper.add_item(40): [10, 20, 30, 40]


### Dynamic Attribute Handling

One of the key features of `DynamicWrapper` is its ability to handle dynamically changing wrapped objects:

In [22]:
# Create a wrapper
class DynamicAttrWrapper(DynamicWrapper):
    _wrapped_map_ = ["obj"]

wrapper = DynamicAttrWrapper()
wrapper.obj = SimpleObject(10)

print("Initial wrapper with SimpleObject:")
print(f"  wrapper.value: {wrapper.value}")
print(f"  wrapper.name: {wrapper.name}")

# Dynamically replace the wrapped object
print("\nReplacing wrapped object with ComplexObject:")
wrapper.obj = ComplexObject([1, 2, 3])

# Now different attributes are available
print("Accessing new attributes:")
print(f"  wrapper.items: {wrapper.items}")
print(f"  wrapper.name: {wrapper.name}")

# Try to access attribute from previous object (should fail)
print("\nTrying to access attribute from previous object:")
try:
    value = wrapper.value
    print(f"  Value: {value}")
except AttributeError:
    print(f"  AttributeError: attribute no longer exists")

Initial wrapper with SimpleObject:
  wrapper.value: 10
  wrapper.name: SimpleObject

Replacing wrapped object with ComplexObject:
Accessing new attributes:
  wrapper.items: [1, 2, 3]
  wrapper.name: ComplexObject

Trying to access attribute from previous object:
  AttributeError: attribute no longer exists


### Using _setattr Method

The `DynamicWrapper` class provides a `_setattr` method that allows you to set attributes directly on the wrapper, bypassing the dynamic attribute resolution:

In [23]:
# Create a wrapper
wrapper = MyWrapper()
wrapper.obj1 = SimpleObject(10)
wrapper.obj2 = ComplexObject([1, 2, 3])

# Set an attribute that exists in a wrapped object
print(f"Initial wrapper.value: {wrapper.value}")
wrapper._setattr("value", "direct_set")
print(f"After wrapper._setattr('value', 'direct_set'):")
print(f"  wrapper.value: {wrapper.value}")  # From wrapper
print(f"  wrapper.obj1.value: {wrapper.obj1.value}")  # From wrapped object (unchanged)

Initial wrapper.value: 10
After wrapper._setattr('value', 'direct_set'):
  wrapper.value: direct_set
  wrapper.obj1.value: 10


### Nested Wrappers

You can also nest wrappers, with one wrapper wrapping another wrapper:

In [24]:
class NestedWrapper(DynamicWrapper):
    """A wrapper that wraps another wrapper."""
    _wrapped_map_ = ["inner"]

# Create an inner wrapper
inner = MyWrapper()
inner.obj1 = SimpleObject(10)
inner.obj2 = ComplexObject([1, 2, 3])

# Create an outer wrapper that wraps the inner wrapper
outer = NestedWrapper()
outer.inner = inner
outer.nested_attr = "from outer"

print("Created nested wrappers:")
print(f"  outer.value: {outer.value}")  # From inner.obj1
print(f"  outer.items: {outer.items}")  # From inner.obj2
print(f"  outer.nested_attr: {outer.nested_attr}")  # From outer

Created nested wrappers:
  outer.value: 10
  outer.items: [1, 2, 3]
  outer.nested_attr: from outer


## Examples

Let's explore some practical examples of how `DynamicWrapper` can be used in real-world scenarios.

### Example 1: Creating a Unified Interface

One common use case for `DynamicWrapper` is to create a unified interface for different types of objects:

In [25]:
class DataSource:
    """A base class for data sources."""
    def get_data(self):
        """Get data from the source."""
        raise NotImplementedError("Subclasses must implement get_data")

class FileDataSource(DataSource):
    """A data source that reads from a file."""
    def __init__(self, filename):
        self.filename = filename
        self.data = f"Data from file: {filename}"
    
    def get_data(self):
        return self.data

class DatabaseDataSource(DataSource):
    """A data source that reads from a database."""
    def __init__(self, connection_string):
        self.connection_string = connection_string
        self.data = f"Data from database: {connection_string}"
    
    def get_data(self):
        return self.data

class APIDataSource(DataSource):
    """A data source that reads from an API."""
    def __init__(self, url):
        self.url = url
        self.data = f"Data from API: {url}"
    
    def get_data(self):
        return self.data
    
    def get_metadata(self):
        return {"source": "API", "url": self.url}

# Create a wrapper that can work with any data source
class DataSourceWrapper(DynamicWrapper):
    _wrapped_map_ = ["source"]
    
    def __init__(self, source=None):
        super().__init__()
        self.source = source
    
    def get_source_type(self):
        """Get the type of the data source."""
        if isinstance(self.source, FileDataSource):
            return "File"
        elif isinstance(self.source, DatabaseDataSource):
            return "Database"
        elif isinstance(self.source, APIDataSource):
            return "API"
        else:
            return "Unknown"

# Create data sources
file_source = FileDataSource("data.csv")
db_source = DatabaseDataSource("mysql://localhost/mydb")
api_source = APIDataSource("https://api.example.com/data")

# Create wrappers
file_wrapper = DataSourceWrapper(file_source)
db_wrapper = DataSourceWrapper(db_source)
api_wrapper = DataSourceWrapper(api_source)

# Use the wrappers
print("Using data source wrappers:")
print(f"  File source: {file_wrapper.get_source_type()}, Data: {file_wrapper.get_data()}")
print(f"  DB source: {db_wrapper.get_source_type()}, Data: {db_wrapper.get_data()}")
print(f"  API source: {api_wrapper.get_source_type()}, Data: {api_wrapper.get_data()}")

# Access API-specific method
try:
    metadata = api_wrapper.get_metadata()
    print(f"  API metadata: {metadata}")
except AttributeError:
    print("  get_metadata not available")

# Switch sources at runtime
print("\nSwitching sources at runtime:")
file_wrapper.source = api_source
print(f"  New source type: {file_wrapper.get_source_type()}")
print(f"  New data: {file_wrapper.get_data()}")
print(f"  Metadata now available: {file_wrapper.get_metadata()}")

Using data source wrappers:
  File source: File, Data: Data from file: data.csv
  DB source: Database, Data: Data from database: mysql://localhost/mydb
  API source: API, Data: Data from API: https://api.example.com/data
  API metadata: {'source': 'API', 'url': 'https://api.example.com/data'}

Switching sources at runtime:
  New source type: API
  New data: Data from API: https://api.example.com/data
  Metadata now available: {'source': 'API', 'url': 'https://api.example.com/data'}


### Example 2: Performance Comparison

Let's compare the performance of `DynamicWrapper` with direct attribute access:

In [26]:
import time

# Create objects
simple = SimpleObject(10)

# Create a wrapper
class PerfWrapper(DynamicWrapper):
    _wrapped_map_ = ["obj"]

wrapper = PerfWrapper()
wrapper.obj = simple

# Measure direct access performance
iterations = 100000
print(f"Running {iterations} iterations for each test...")

# Direct access
start_time = time.time()
for _ in range(iterations):
    value = simple.value
    simple.value = value + 1
direct_time = time.time() - start_time

# Reset value
simple.value = 10

# Wrapper access
start_time = time.time()
for _ in range(iterations):
    value = wrapper.value
    wrapper.value = value + 1
wrapper_time = time.time() - start_time

print(f"Direct access time: {direct_time:.6f} seconds")
print(f"Wrapper access time: {wrapper_time:.6f} seconds")
print(f"Ratio (wrapper/direct): {wrapper_time/direct_time:.2f}x slower")

Running 100000 iterations for each test...
Direct access time: 0.013430 seconds
Wrapper access time: 2.060614 seconds
Ratio (wrapper/direct): 153.44x slower


## API Highlights

The `DynamicWrapper` class provides the following key components:

```python
class DynamicWrapper(BaseObject):
    """An object that can call the attributes/functions of embedded objects, acting as if it is inheriting from them."""
    
    # Class Attributes
    _wrapped_map_: list[str] = []  # List of attribute names containing objects to wrap
    
    # Magic Methods
    def __getattr__(self, name: str) -> Any:
        """Gets the attribute of another object if that attribute is not present in this object."""
        
    def __setattr__(self, name: str, value: Any) -> None:
        """Sets the attribute of another object if that attribute name is not present in this object."""
        
    def __delattr__(self, name: str) -> None:
        """Deletes the attribute of another object if that attribute name is not present in this object."""
    
    # Instance Methods
    def _setattr(self, name: str, value: Any):
        """An override method that will set an attribute of this object without checking its presence in other objects."""
```

For more details, refer to the full API documentation.

## Troubleshooting / FAQs

### Q: Why is my attribute not being found in the wrapped objects?

A: Check the following:
- Make sure the attribute exists in at least one of the wrapped objects
- Verify that the wrapped objects are correctly assigned to the attributes listed in `_wrapped_map_`
- Check the order of attributes in `_wrapped_map_`, as it determines the lookup order
- Ensure you're not accidentally shadowing the attribute in the wrapper itself

### Q: Why is DynamicWrapper slower than direct attribute access?

A: `DynamicWrapper` performs dynamic attribute resolution for every attribute access, which involves checking multiple objects. This overhead makes it slower than direct attribute access. If performance is critical, consider using `StaticWrapper` instead, which creates property descriptors at class definition time.

### Q: How do I set an attribute directly on the wrapper?

A: Use the `_setattr` method to set an attribute directly on the wrapper, bypassing the dynamic attribute resolution:

```python
wrapper._setattr("attribute_name", value)
```

### Q: Can I wrap objects of different types?

A: Yes, `DynamicWrapper` can wrap objects of any type. This is one of its key advantages - it can handle heterogeneous objects and dynamically changing objects.

### Q: How do I handle attribute name conflicts between wrapped objects?

A: Attributes are resolved in the order specified in `_wrapped_map_`. If multiple wrapped objects have the same attribute, the one from the object listed first in `_wrapped_map_` will be used.

### Comparison with StaticWrapper

The baseobjects package includes another wrapper class called `StaticWrapper`, which has a different approach to wrapping objects. Let's compare the two:

**DynamicWrapper advantages:**
1. Can handle dynamically changing wrapped objects
2. No need to call _wrap() after changing wrapped objects
3. More flexible with attribute resolution
4. Simpler to use for quick prototyping

**DynamicWrapper disadvantages:**
1. Slower performance (typically 4.4x slower than direct access)
2. No IDE auto-completion for wrapped object attributes
3. Less explicit about which attributes are available

**When to use DynamicWrapper:**
- When wrapped objects change frequently during runtime
- When you need to wrap various indeterminate object types
- When performance is not a critical concern
- For quick prototyping and development

## Conclusion and Next Steps

In this tutorial, we've explored the `DynamicWrapper` class and its capabilities for wrapping objects and providing dynamic attribute resolution. We've learned how to create basic and custom wrappers, access wrapped object attributes and methods, handle dynamic attribute changes, and use advanced features like nested wrappers.

The `DynamicWrapper` class provides a flexible way to create unified interfaces for different types of objects, especially when those objects might change during runtime. While it's not as performant as direct attribute access, it offers significant flexibility and simplicity for many use cases.

### Next Steps

- Explore the `StaticWrapper` class, which provides better performance at the cost of some flexibility
- Create your own custom wrapper classes for specific use cases
- Experiment with combining `DynamicWrapper` with other baseobjects components
- Check out the examples directory for more examples of using wrappers

Remember that `DynamicWrapper` is best used when:
- Wrapped objects change frequently during runtime
- You need to wrap various indeterminate object types
- Performance is not a critical concern
- You're doing quick prototyping and development